In [ ]:
!pip install gymnasium[atari]
!pip install ale-py

In [ ]:
import os
import random
import numpy as np
import gymnasium as gym
import cv2
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D, Flatten, Dense, Input, Add, Concatenate
from tensorflow.keras.optimizers import Adam
from collections import deque
import gc
import matplotlib.pyplot as plt

# --- CONFIGURAÇÃO DO ATARI ---
import ale_py
gym.register_envs(ale_py)

# --- HIPERPARÂMETROS DO PLANO B ---
ENV_NAME = "ALE/MsPacman-v5"
DRIVE_PATH = "/content/drive/MyDrive/dueling_ddqn_mspacman_hybrid.weights.h5"

EPISODE_START = 13330    # Ponto de partida do reset do Plano B
EPISODES = 14000
BATCH_SIZE = 128
MEMORY_SIZE = 30000
GAMMA = 0.99
LEARNING_RATE = 0.00025

EPSILON = 0.02
EPSILON_MIN = 0.02
EPSILON_DECAY = 1

print("--- [SISTEMA] CONECTANDO AO GOOGLE DRIVE ---")
from google.colab import drive
drive.mount('/content/drive')

env = gym.make(ENV_NAME, repeat_action_probability=0.25)
action_space_size = int(env.action_space.n)

def preprocess_frame(frame):
    frame = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    frame = cv2.resize(frame, (84, 84))
    return (frame / 255.0).astype(np.float32)

# --- MAPEAMENTO DA RAM DO ATARI (Ms. Pacman) ---
def get_hybrid_features(env_instance):
    ram = env_instance.unwrapped.ale.getRAM()
    # Posições X e Y clássicas da Ms. Pacman na RAM do emulador
    pacman_x = ram[10] / 255.0
    pacman_y = ram[16] / 255.0
    # Posições dos fantasmas (facilitando a fuga nas quinas)
    blinky_x = ram[6] / 255.0
    pinky_x = ram[7] / 255.0
    inkey_x = ram[8] / 255.0
    sue_x = ram[9] / 255.0

    return np.array([pacman_x, pacman_y, blinky_x, pinky_x, inkey_x, sue_x], dtype=np.float32)

# --- ARQUITETURA HÍBRIDA MULTI-INPUT ---
def build_hybrid_dueling_dqn(img_shape, feature_shape, action_space):
    # Ramo 1: Imagem (CNN)
    img_input = Input(shape=img_shape, name="image_input")
    x = Conv2D(32, (8, 8), strides=4, activation="relu")(img_input)
    x = Conv2D(64, (4, 4), strides=2, activation="relu")(x)
    x = Conv2D(64, (3, 3), strides=1, activation="relu")(x)
    conv_output = Flatten()(x)

    # Ramo 2: Coordenadas Numéricas (RAM)
    feat_input = Input(shape=feature_shape, name="feature_input")

    # Fusão dos dois mundos
    merged = Concatenate()([conv_output, feat_input])

    # Fluxo Dueling sobre o vetor fundido
    value_fc = Dense(512, activation="relu")(merged)
    value = Dense(1)(value_fc)

    advantage_fc = Dense(512, activation="relu")(merged)
    advantage = Dense(action_space)(advantage_fc)

    mean_advantage = tf.keras.layers.Lambda(lambda x: tf.reduce_mean(x, axis=1, keepdims=True))(advantage)
    centered_advantage = tf.keras.layers.Subtract()([advantage, mean_advantage])
    outputs = Add()([value, centered_advantage])

    model = Model(inputs=[img_input, feat_input], outputs=outputs)
    return model

input_image_shape = (84, 84, 4)
input_feature_shape = (6,)

model = build_hybrid_dueling_dqn(input_image_shape, input_feature_shape, action_space_size)
target_model = build_hybrid_dueling_dqn(input_image_shape, input_feature_shape, action_space_size)
target_model.set_weights(model.get_weights())

optimizer = Adam(learning_rate=LEARNING_RATE)
model.compile(optimizer=optimizer, loss="huber")

if os.path.exists(DRIVE_PATH):
    model.load_weights(DRIVE_PATH)
    target_model.set_weights(model.get_weights())
    print("--- [SUCESSO] Pesos HÍBRIDOS carregados do Drive! ---")
else:
    print("--- [INFO] Inicializando cérebro híbrido do zero... ---")

@tf.function
def train_step(img_states, feat_states, targets):
    with tf.GradientTape() as tape:
        predictions = model({"image_input": img_states, "feature_input": feat_states}, training=True)
        loss = model.compute_loss({"image_input": img_states, "feature_input": feat_states}, targets, predictions)
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

replay_buffer = deque(maxlen=MEMORY_SIZE)
historico_rewards_reais = []

print("--- [TREINO] Iniciando Plano B: Fusão Visual-Numérica Anti-Trancamento ---")

for episode in range(EPISODE_START, EPISODES + 1):
    state, info = env.reset()
    state_deque = deque(maxlen=4)
    frame = preprocess_frame(state)
    for _ in range(4):
        state_deque.append(frame)

    done = False
    pontuacao_real = 0
    steps = 0
    last_score_step = 0
    vidas_anterior = info.get("lives", 3)

    while not done:
        steps += 1
        state_stacked = np.stack(state_deque, axis=-1)
        state_input_img = np.expand_dims(state_stacked, axis=0)

        # Extrai vetor de coordenadas atual da RAM
        state_input_feat = np.expand_dims(get_hybrid_features(env), axis=0)

        # 1. TOMADA DE DECISÃO PADRÃO HÍBRIDA
        if random.random() < EPSILON:
            action = env.action_space.sample()
        else:
            q_values = model({"image_input": state_input_img, "feature_input": state_input_feat}, training=False)
            action = np.argmax(q_values[0])

        # 2. QUEBRA DE INÉRCIA ADAPTATIVA
        travado_na_quina = (steps - last_score_step) > 40
        if travado_na_quina and random.random() < 0.4:
            action = env.action_space.sample()

        # 3. EXECUÇÃO
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        pontuacao_real += reward

        if reward > 0:
            last_score_step = steps

        # 4. REWARD SHAPING
        if travado_na_quina:
            reward -= 2.0

        if reward >= 200:
            reward = (reward * 1.5) + 20.0
        elif reward == 0:
            reward -= 0.1
        elif steps % 10 == 0 and (steps - last_score_step) < 30:
            reward += 0.5

        if pontuacao_real >= 2300 and steps % 20 == 0:
            reward += 15.0

        vidas_atuais = info.get("lives", 3)
        if vidas_atuais < vidas_anterior:
            reward -= 50.0
            vidas_anterior = vidas_atuais
            next_frame = preprocess_frame(next_state)
            state_deque.clear()
            for _ in range(4):
                state_deque.append(next_frame)

        # 5. ARMAZENAMENTO NA MEMÓRIA COM AS COMPONENTES EXTRAS
        next_frame = preprocess_frame(next_state)
        next_feat = get_hybrid_features(env)
        # Salvamos o estado de features numéricas atual na tupla do buffer
        replay_buffer.append((list(state_deque), state_input_feat[0], action, reward, next_frame, next_feat, done))
        state_deque.append(next_frame)

        # 6. APRENDIZADO EM MINIBATCH MULTI-INPUT
        if len(replay_buffer) > BATCH_SIZE and steps % 4 == 0:
            minibatch = random.sample(replay_buffer, BATCH_SIZE)

            img_b = np.array([np.stack(t[0], axis=-1) for t in minibatch], dtype=np.float32)
            feat_b = np.array([t[1] for t in minibatch], dtype=np.float32)
            actions_b = np.array([t[2] for t in minibatch], dtype=np.int32)
            rewards_b = np.array([t[3] for t in minibatch], dtype=np.float32)

            next_img_b = []
            for t in minibatch:
                hist = t[0][1:] + [t[4]]
                next_img_b.append(np.stack(hist, axis=-1))
            next_img_b = np.array(next_img_b, dtype=np.float32)
            next_feat_b = np.array([t[5] for t in minibatch], dtype=np.float32)

            dones_b = np.array([t[6] for t in minibatch], dtype=np.float32)

            # Predições Target Híbridas
            main_next_q = model({"image_input": next_img_b, "feature_input": next_feat_b}, training=False).numpy()
            best_actions = np.argmax(main_next_q, axis=1)
            target_next_q = target_model({"image_input": next_img_b, "feature_input": next_feat_b}, training=False).numpy()

            targets = model({"image_input": img_b, "feature_input": feat_b}, training=False).numpy()
            for i in range(BATCH_SIZE):
                if dones_b[i]:
                    targets[i][actions_b[i]] = rewards_b[i]
                else:
                    targets[i][actions_b[i]] = rewards_b[i] + GAMMA * target_next_q[i][best_actions[i]]

            train_step(img_b, feat_b, targets)

    if EPSILON > EPSILON_MIN:
        EPSILON *= EPSILON_DECAY

    if episode % 10 == 0:
        target_model.set_weights(model.get_weights())

    historico_rewards_reais.append(pontuacao_real)
    print(f"Episódio {episode}/{EPISODES} | Score Real do Jogo={pontuacao_real:.1f} | Epsilon={EPSILON:.4f}")

    if episode % 5 == 0:
        gc.collect()

    if episode % 50 == 0:
        model.save_weights(DRIVE_PATH)
        print(f"\n--- [SEGURANÇA] Novo Checkpoint Caçador HÍBRIDO salvo no Drive! ---\n")
        tf.keras.backend.clear_session()
        gc.collect()

env.close()

--- [SISTEMA] CONECTANDO AO GOOGLE DRIVE ---
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--- [SUCESSO] Pesos HÍBRIDOS carregados do Drive! ---
--- [TREINO] Iniciando Plano B: Fusão Visual-Numérica Anti-Trancamento ---
Episódio 13330/14000 | Score Real do Jogo=860.0 | Epsilon=0.0200
Episódio 13331/14000 | Score Real do Jogo=1810.0 | Epsilon=0.0200
Episódio 13332/14000 | Score Real do Jogo=1490.0 | Epsilon=0.0200
Episódio 13333/14000 | Score Real do Jogo=1240.0 | Epsilon=0.0200
Episódio 13334/14000 | Score Real do Jogo=1130.0 | Epsilon=0.0200
Episódio 13335/14000 | Score Real do Jogo=2090.0 | Epsilon=0.0200
Episódio 13336/14000 | Score Real do Jogo=3660.0 | Epsilon=0.0200
Episódio 13337/14000 | Score Real do Jogo=2450.0 | Epsilon=0.0200
Episódio 13338/14000 | Score Real do Jogo=1790.0 | Epsilon=0.0200
Episódio 13339/14000 | Score Real do Jogo=1100.0 | Epsilon=0.0200
Episódio 13340/14000 | Score Real do J